# Other optimizers: PSO and QNG

[`06a`](06a_variational_training.ipynb) trained a QAOA with Differential Evolution. Polypus brings two other optimizers for the same `train()`: PSO, which barely changes the call, and QNG, which needs an extra piece of its own for the circuit. This notebook trains the same MaxCut problem with both.

## The same circuit

The one-layer QAOA on the same three-node path graph from [`06a`](06a_variational_training.ipynb) is reused, with the best possible cut already computed there by brute force:

In [1]:
import polypus

Param = polypus.Param
edges = [(0, 1), (1, 2)]


def cut_value(bitstring):
    bits = bitstring[::-1]
    return sum(1 for i, j in edges if bits[i] != bits[j])


qc = polypus.Circuit(3)
qc.h(0).h(1).h(2)
qc.rzz(0, 1, Param(0))
qc.rzz(1, 2, Param(0))
qc.rx(0, Param(1)).rx(1, Param(1)).rx(2, Param(1))
qc.measure_all()

best_cut = 2

## PSO: the same API, a different optimizer

Particle Swarm Optimization, PSO, keeps a swarm of particles that move through the parameter space, drawn toward their own best result and toward the whole swarm's best. The call to `train()` is identical to the one in [`06a`](06a_variational_training.ipynb): same circuit, same infrastructure arguments, same `expectation_function`. The only thing that changes is `polypus.DE(...)` becoming `polypus.PSO(...)`:

In [2]:
pso = polypus.PSO(generations=60, population_size=20, tolerance=0.001, seed=7)

result_pso = polypus.train(
    qc,
    pso,
    shots=1000,
    n_qpus=1,
    dimensions=qc.num_params,
    expectation_function=cut_value,
    infrastructure="local",
    nodes=1,
    cores_per_qpu=2,
    id="maxcut-pso",
)
print(result_pso.best_params)
print(result_pso.best_fitness)
print(result_pso.iterations_run)
print(result_pso.converged)

[-1.0443325989004881, 0.7730998724246112]
1.679
60
False


`iterations_run` reaches the full 60 and `converged` comes out `False`: unlike the Differential Evolution in [`06a`](06a_variational_training.ipynb), which stopped early once it stabilized, PSO uses up the whole budget without dropping below `tolerance`, even though the `best_fitness` found is practically the same. Evaluated the same way as in [`06a`](06a_variational_training.ipynb), binding `best_params` and measuring:

In [3]:
final_qc_pso = polypus.Circuit.from_qasm2(qc.to_qasm2(result_pso.best_params))
result_pso_run = polypus.run_quantum_circuit(
    final_qc_pso, shots=2000, infrastructure="local"
)
counts_pso = result_pso_run.counts[0]
total_pso = sum(counts_pso.values())
average_cut_pso = sum(cut_value(bits) * n for bits, n in counts_pso.items()) / total_pso

print("measured average cut:", average_cut_pso)
print("best possible cut:", best_cut)

measured average cut: 1.665
best possible cut: 2


A result practically the same as PSO's.

## QNG: the same circuit, an extra piece

Quantum Natural Gradient, QNG, also follows the cost function's gradient, but corrects it with the circuit's own parameter-space geometry, instead of treating it as a flat space. In exchange for that correction, it asks for something neither DE nor PSO ask for: `variance_function`, a function that estimates how much the result of measuring each parameter's generator varies at the current point. Polypus cannot compute it alone because it depends on each circuit's specific structure.

For parameter `beta`, index 1, that generator is the sum of `X` on each qubit, the same rotation the mixing layer applies. Its variance is estimated by building the circuit up to right before that layer, with `gamma` already fixed at its current value, rotating to the `X` basis with an H before measuring, and computing the variance of the sum of the measured signs. For parameter `gamma`, index 0, there is no previous layer to fix: by convention, its variance is taken as 0, and `tikhonov_reg`'s regularization takes care of that not causing a division by too small a number:

In [4]:
def variance_fn(theta, a):
    if a == 0:
        return 0.0

    partial_qc = polypus.Circuit(3)
    partial_qc.h(0).h(1).h(2)
    partial_qc.rzz(0, 1, theta[0])
    partial_qc.rzz(1, 2, theta[0])
    partial_qc.h(0).h(1).h(2)  # rotate to the X basis before measuring
    partial_qc.measure_all()

    # fixed seed: the outer train() has its own seed, but this call is
    # independent, and without one the result would not be reproducible
    result = polypus.run_quantum_circuit(
        partial_qc, shots=2000, infrastructure="local", seed=7
    )
    counts = result.counts[0]
    total = sum(counts.values())

    exp_h = 0.0
    exp_h2 = 0.0
    for bitstring, n in counts.items():
        prob = n / total
        bits = bitstring[::-1]
        value = sum(1 if bits[i] == "0" else -1 for i in range(3))
        exp_h += value * prob
        exp_h2 += (value**2) * prob

    return exp_h2 - exp_h**2

`polypus.QNG` takes `variance_function` as its first argument, the new piece neither `DE` nor `PSO` have. In exchange, it has no `tolerance` parameter at all: QNG has no early-stopping criterion, so `converged` always comes out `False`, whether or not it improved in the last iterations. The rest of `train()` stays the same:

In [5]:
qng = polypus.QNG(
    variance_fn,
    max_iters=40,
    learning_rate=0.1,
    finite_difference_step=0.1,
    tikhonov_reg=0.05,
    seed=7,
)

result_qng = polypus.train(
    qc,
    qng,
    shots=1000,
    n_qpus=1,
    dimensions=qc.num_params,
    expectation_function=cut_value,
    infrastructure="local",
    nodes=1,
    cores_per_qpu=2,
    id="maxcut-qng",
)
print(result_qng.best_params)
print(result_qng.best_fitness)
print(result_qng.converged)

[-5.11110305724973, -0.5253962588960064]
1.529
False


Evaluated the same way as the previous two:

In [6]:
final_qc_qng = polypus.Circuit.from_qasm2(qc.to_qasm2(result_qng.best_params))
result_qng_run = polypus.run_quantum_circuit(
    final_qc_qng, shots=2000, infrastructure="local"
)
counts_qng = result_qng_run.counts[0]
total_qng = sum(counts_qng.values())
average_cut_qng = sum(cut_value(bits) * n for bits, n in counts_qng.items()) / total_qng

print("measured average cut:", average_cut_qng)
print("best possible cut:", best_cut)

measured average cut: 1.5465
best possible cut: 2


QNG's average cut lands somewhat below DE's and PSO's, with this unadjusted configuration of `learning_rate` and `finite_difference_step`: the point of this notebook is showing the API, not squeezing every last bit out of each optimizer.

## Summary

This notebook trained the same QAOA from [`06a`](06a_variational_training.ipynb) with two more optimizers: PSO, with the same call to `train()` except for the object passed in `method`, and QNG, which also asks for a `variance_function` of its own for the circuit to correct the gradient with the parameter space's geometry.

## End of the series

This series has walked the full path: from the first circuit to training one with data. Readers who want to go further can check `examples/`, with bigger cases, including a full MaxCut with QAOA example, and the main README, which documents performance benchmarks across backends.

Back to the index: [`00_welcome.ipynb`](00_welcome.ipynb).